# 08 - Model Persistence

**Goal for today:** take the winning model from notebook 07, retrain it properly for
production, save it to disk, and prove it can be loaded and used completely outside
this notebook - the first real step toward deployment.

**Why this matters for MLA-C01 (bridging Domain 2 into Domain 3):**
Every model we've trained so far has lived only inside a Jupyter kernel - the moment
you close the notebook, it's gone, and all that training work has to happen again
from scratch. A model that only exists inside a notebook can't be deployed to
**SageMaker**, can't be called by an application, and can't serve real predictions.
Model persistence - saving trained models to disk in a reusable format - is the
literal bridge between "a model I trained" and "a model I can deploy."

## Step 0: Load data and pick our winning configuration

**What this cell does:** loads the processed dataset one more time, and hardcodes
the winning hyperparameters `RandomizedSearchCV` found back in notebook 07. We're not
re-running the search here - that's a one-time, exploratory step. Once you've found
good settings, you write them down and move on; re-searching every time you retrain
would be wasteful and, worse, could give you a *different* answer each run.

In [1]:
import pandas as pd

df = pd.read_csv('../data/telco_churn_processed.csv')

X = df.drop(columns=['Churn'])
y = df['Churn']

# Winning hyperparameters found via RandomizedSearchCV in notebook 07
best_params = {
    'n_estimators': 400,
    'max_depth': 10,
    'min_samples_split': 5,
    'min_samples_leaf': 4,
    'max_features': 'log2',
    'random_state': 42,
}

X.shape

(7043, 30)

## Step 1: One last honest check on a proper holdout split

**What this cell does:** before we commit to this configuration for production, we
do one final train/test evaluation - the same pattern as every prior notebook - purely
as a last sanity check that this model still performs as expected. This isn't new
information, just confirmation before we move to the next step.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

check_model = RandomForestClassifier(**best_params)
check_model.fit(X_train, y_train)

check_pred = check_model.predict(X_test)
check_proba = check_model.predict_proba(X_test)[:, 1]

print('accuracy: ', accuracy_score(y_test, check_pred))
print('precision:', precision_score(y_test, check_pred))
print('recall:   ', recall_score(y_test, check_pred))
print('f1:       ', f1_score(y_test, check_pred))
print('roc_auc:  ', roc_auc_score(y_test, check_proba))

accuracy:  0.8034066713981547
precision: 0.6701754385964912
recall:    0.5106951871657754
f1:        0.5796661608497724
roc_auc:   0.8439910615102431


## Step 2: Retrain on ALL available data for production

**This is an important, easy-to-miss concept.** Every model up to this point has
deliberately trained on only 80% of the data, holding back 20% to honestly measure
performance. That was the right call *while we were still deciding* which algorithm
and settings to use.

But now that we've made that decision and confirmed it works, holding back 20% of
real data serves no further purpose - it just means our final, deployed model is
learning from less information than it could be. **The standard practice, once model
selection is finalized, is to retrain on 100% of the available data using the chosen
configuration**, since we're no longer trying to *evaluate* this exact model - we
already did that in Step 1 and in notebook 07 - we're now trying to make the best
possible production model to actually deploy.

This is a subtle but real distinction the exam expects you to understand: the model
you *evaluate* and the model you *ship* are often trained on different amounts of
data, and that's intentional, not an inconsistency.

In [3]:
final_model = RandomForestClassifier(**best_params)
final_model.fit(X, y)

print("Final model trained on all", X.shape[0], "rows.")

Final model trained on all 7043 rows.


## Step 3: Save the model to disk

**What this cell does:** `joblib` is scikit-learn's recommended library for saving
trained models - it's more efficient than Python's general-purpose `pickle` for
objects containing large numpy arrays, which is exactly what a trained Random
Forest is under the hood (recall it's literally hundreds of decision trees).

`joblib.dump(model, filepath)` serializes the entire trained model - every tree,
every learned split - into a single file that can be loaded back later without
needing to retrain anything.

We're saving this into a new top-level `models/` folder, not `data/` or `notebooks/`
- keeping trained model artifacts clearly separated from raw data and exploratory
code is a real project-organization habit, and it's exactly the folder Terraform will
later reference when we set up S3 storage for this model.

In [4]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(final_model, '../models/churn_model.joblib')
print("Model saved to ../models/churn_model.joblib")

Model saved to ../models/churn_model.joblib


## Step 4: Also save the exact column order

**Why this matters, and it's a genuinely easy mistake to make in production:** our
model expects input data with a specific number of columns, in a specific order,
matching exactly how `X` was structured when we trained it (remember all that one-hot
encoding from notebook 04). If someone - including future-you - tries to feed this
model new customer data with columns in a different order, or a column missing, the
model won't necessarily error out - it might silently produce garbage predictions,
because it has no way of knowing "column 5" is supposed to mean `TotalCharges` versus
anything else.

Saving the exact expected column list alongside the model is a small step that
prevents a very real, very common production bug.

In [5]:
feature_columns = X.columns.tolist()

joblib.dump(feature_columns, '../models/feature_columns.joblib')
print(f"Saved {len(feature_columns)} expected feature columns.")

Saved 30 expected feature columns.


## Step 5: Prove it works - reload everything as if from a fresh start

**What this cell does:** this is the real test. We deliberately load the saved model
and column list back in - exactly as a separate script, application, or SageMaker
endpoint would - without relying on any variable already sitting in this notebook's
memory. If this works, we've genuinely proven the model can live outside this
notebook, not just inside it.

In [6]:
loaded_model = joblib.load('../models/churn_model.joblib')
loaded_columns = joblib.load('../models/feature_columns.joblib')

print("Model loaded:", type(loaded_model).__name__)
print("Number of expected columns:", len(loaded_columns))

Model loaded: RandomForestClassifier
Number of expected columns: 30


## Step 6: Run a real prediction through the reloaded model

**What this cell does:** grabs a single real customer's data from our dataset (row
0), makes sure its columns are in the exact order `loaded_columns` expects using
`.reindex()`, and runs it through the freshly reloaded model. `.reindex(columns=...)`
reorders (and would fill in as missing, if needed) columns to match a given list -
the practical enforcement of the column-order safeguard from Step 4.

In [7]:
sample_customer = X.iloc[[0]].reindex(columns=loaded_columns)

prediction = loaded_model.predict(sample_customer)[0]
probability = loaded_model.predict_proba(sample_customer)[0][1]

print(f"Predicted churn: {'Yes' if prediction == 1 else 'No'}")
print(f"Churn probability: {probability:.2%}")
print(f"Actual churn label for this customer: {'Yes' if y.iloc[0] == 1 else 'No'}")

Predicted churn: Yes
Churn probability: 60.43%
Actual churn label for this customer: No


If this ran without error and gave you a sensible-looking prediction, you've just
confirmed the entire loop: **train once, save once, load anywhere, predict on demand.**
That is genuinely the core mechanic behind how a deployed ML model works in
production - a SageMaker endpoint is, at its heart, this exact same
load-model-then-predict pattern, just running inside AWS infrastructure instead of a
notebook cell.

---

**That's it for today.** Small, complete increment:
- retrained our winning Random Forest configuration on 100% of the data for
  production, understanding why that differs from the evaluation model
- saved the trained model to disk with `joblib`, in a new `models/` folder
- saved the exact expected column order alongside it, to guard against a real
  production bug
- reloaded both from disk and successfully generated a prediction, proving the model
  works independently of this notebook

**Next session:** we'll pull this logic out of notebooks entirely and into a proper
Python script in `src/` - the step that turns "code I ran by hand" into "code a
pipeline can run automatically" - setting us up to move into actual AWS
infrastructure with Terraform shortly after.